# 03b — Baseline Model (80/20 Design)

**Parallel to `03_baseline_model.ipynb` — identical logic, different data.**

**Inputs:** `_80` preprocessed files from `02b_data_preprocessing_8020.ipynb`

| File | Patients | Description |
|---|---|---|
| `X_train_80.csv` | 2,016 | 80% eICU — scaler fit on these |
| `X_val_80.csv`   |   504 | 20% eICU — transformed |
| `X_mimic_80.csv` |   136 | MIMIC external test |

**Outputs** (new names, nothing overwritten):
- `models/rf_calibrated_80.pkl`
- `models/rf_final_80.pkl`
- `models/model_params_80.json`

**Key difference from 03:** no internal eICU test set — MIMIC is the sole test. GridSearch is rerun on the larger training set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q imbalanced-learn shap xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, pickle, json
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.calibration     import CalibratedClassifierCV
from xgboost                 import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics         import (roc_auc_score, average_precision_score,
                                     roc_curve, confusion_matrix)
from imblearn.over_sampling  import SMOTE
from imblearn.pipeline       import Pipeline as ImbPipeline
from collections             import Counter

SEED = 42
np.random.seed(SEED)

BASE       = '/content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed'
MODEL_DIR  = '/content/drive/MyDrive/AI in Medicine/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 1. Load 80/20 Preprocessed Data

In [ ]:
X_train = pd.read_csv(f'{BASE}/X_train_80.csv')
y_train = pd.read_csv(f'{BASE}/y_train_80.csv').squeeze()
X_val   = pd.read_csv(f'{BASE}/X_val_80.csv')
y_val   = pd.read_csv(f'{BASE}/y_val_80.csv').squeeze()
X_mimic = pd.read_csv(f'{BASE}/X_mimic_80.csv')
y_mimic = pd.read_csv(f'{BASE}/y_mimic_80.csv').squeeze()

print(f'X_train : {X_train.shape}   mortality={y_train.mean():.3f}')
print(f'X_val   : {X_val.shape}     mortality={y_val.mean():.3f}')
print(f'X_mimic : {X_mimic.shape}   mortality={y_mimic.mean():.3f}')
print(f'\nClass imbalance (train): {Counter(y_train)}')
print(f'Features: {X_train.shape[1]}')

X_train : (2016, 81)   mortality=0.050
X_val   : (504, 81)     mortality=0.050
X_mimic : (136, 81)   mortality=0.338

Class imbalance (train): Counter({0: 1915, 1: 101})
Features: 81


## 2. Evaluation Framework

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model(model, X, y_true, label='', threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred  = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        'label':       label,
        'ROC-AUC':     roc_auc_score(y_true, proba),
        'PR-AUC':      average_precision_score(y_true, proba),
        'Sensitivity': tp / (tp + fn),
        'Specificity': tn / (tn + fp),
        'threshold':   threshold,
        'proba':       proba,
        'y_true':      np.array(y_true),
    }

def best_threshold(model, X_v, y_v, min_specificity=0.80):
    proba = model.predict_proba(X_v)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_v, proba)
    specificity = 1 - fpr
    mask = specificity >= min_specificity
    if not mask.any():
        return float(thresholds[np.argmax(tpr - fpr)])
    return float(thresholds[mask][np.argmax(tpr[mask])])

print('Evaluation framework ready.')

Evaluation framework ready.


## 3. Logistic Regression

In [ ]:
lr_cw = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
cv_lr = cross_validate(lr_cw, X_train, y_train, cv=cv_strategy,
                       scoring=['roc_auc', 'average_precision'])
print(f'LR (class_weight=balanced) 5-fold CV:')
print(f'  ROC-AUC : {cv_lr["test_roc_auc"].mean():.3f} ± {cv_lr["test_roc_auc"].std():.3f}')
print(f'  PR-AUC  : {cv_lr["test_average_precision"].mean():.3f} ± {cv_lr["test_average_precision"].std():.3f}')

lr_final = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_final.fit(X_train, y_train)
lr_thr = best_threshold(lr_final, X_val, y_val)
lr_val = evaluate_model(lr_final, X_val, y_val, label='LR (val)', threshold=lr_thr)
print(f'\nLR — Val  ROC-AUC={lr_val["ROC-AUC"]:.3f}  '
      f'Sensitivity={lr_val["Sensitivity"]:.3f}  Specificity={lr_val["Specificity"]:.3f}')

LR (class_weight=balanced) 5-fold CV:
  ROC-AUC : 0.775 ± 0.046
  PR-AUC  : 0.273 ± 0.082

LR — Val  ROC-AUC=0.839  Sensitivity=0.720  Specificity=0.850


## 4. Random Forest — GridSearch + Calibration

GridSearch rerun on 2,016 patients (vs 1,512 in notebook 03). Sigmoid calibration fit entirely inside X_train_80 via 5-fold CV — X_val and X_mimic untouched.

In [ ]:
rf_base = RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)
cv_rf   = cross_validate(rf_base, X_train, y_train, cv=cv_strategy,
                         scoring=['roc_auc', 'average_precision'])
print(f'RF (default) 5-fold CV:')
print(f'  ROC-AUC : {cv_rf["test_roc_auc"].mean():.3f} ± {cv_rf["test_roc_auc"].std():.3f}')

RF (default) 5-fold CV:
  ROC-AUC : 0.845 ± 0.038


In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 15, 20],
    'min_samples_leaf': [1, 5, 10],
}
rf_gs = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
    param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1, verbose=1
)
rf_gs.fit(X_train, y_train)
print(f'Best CV AUC : {rf_gs.best_score_:.3f}')
print(f'Best params : {rf_gs.best_params_}')

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best CV AUC : 0.857
Best params : {'max_depth': 15, 'min_samples_leaf': 10, 'n_estimators': 300}


In [ ]:
rf_best_params = rf_gs.best_params_

rf_base_for_cal = RandomForestClassifier(
    **rf_best_params, class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf_calibrated = CalibratedClassifierCV(rf_base_for_cal, method='sigmoid', cv=cv_strategy)
rf_calibrated.fit(X_train, y_train)

rf_final = rf_gs.best_estimator_   # already fitted by GridSearchCV

rf_thr  = best_threshold(rf_final,      X_val, y_val, min_specificity=0.80)
cal_thr = best_threshold(rf_calibrated, X_val, y_val, min_specificity=0.80)

rf_val  = evaluate_model(rf_final,      X_val, y_val, label='RF (val)',            threshold=rf_thr)
cal_val = evaluate_model(rf_calibrated, X_val, y_val, label='RF calibrated (val)', threshold=cal_thr)

print(f'RF uncalibrated — thr={rf_thr:.3f}  AUC={rf_val["ROC-AUC"]:.3f}  '
      f'Sens={rf_val["Sensitivity"]:.3f}  Spec={rf_val["Specificity"]:.3f}')
print(f'RF calibrated   — thr={cal_thr:.3f}  AUC={cal_val["ROC-AUC"]:.3f}  '
      f'Sens={cal_val["Sensitivity"]:.3f}  Spec={cal_val["Specificity"]:.3f}')

RF uncalibrated — thr=0.365  AUC=0.831  Sens=0.640  Spec=0.898
RF calibrated   — thr=0.099  AUC=0.826  Sens=0.640  Spec=0.904


## 5. XGBoost

In [ ]:
neg, pos   = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos  = neg / pos
print(f'scale_pos_weight: {scale_pos:.1f}')

xgb_gs = GridSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos, random_state=SEED,
                  eval_metric='aucpr', verbosity=0),
    {'n_estimators': [100, 200], 'max_depth': [3, 5, 7],
     'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.8, 1.0]},
    cv=cv_strategy, scoring='roc_auc', n_jobs=-1, verbose=1
)
xgb_gs.fit(X_train, y_train)
print(f'XGB best params : {xgb_gs.best_params_}')
print(f'XGB best CV AUC : {xgb_gs.best_score_:.3f}')

xgb_final = xgb_gs.best_estimator_
xgb_thr   = best_threshold(xgb_final, X_val, y_val, min_specificity=0.80)
xgb_val   = evaluate_model(xgb_final, X_val, y_val, label='XGB (val)', threshold=xgb_thr)
print(f'XGB — Val AUC={xgb_val["ROC-AUC"]:.3f}  '
      f'Sens={xgb_val["Sensitivity"]:.3f}  Spec={xgb_val["Specificity"]:.3f}')

scale_pos_weight: 19.0
Fitting 5 folds for each of 36 candidates, totalling 180 fits
XGB best params : {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}
XGB best CV AUC : 0.832
XGB — Val AUC=0.793  Sens=0.560  Spec=0.827


## 6. Model Comparison (Validation Set)

In [ ]:
all_val_results = [lr_val, rf_val, cal_val, xgb_val]

summary = pd.DataFrame([{
    'Model':       r['label'],
    'ROC-AUC':     round(r['ROC-AUC'], 3),
    'PR-AUC':      round(r['PR-AUC'], 3),
    'Sensitivity': round(r['Sensitivity'], 3),
    'Specificity': round(r['Specificity'], 3),
    'Threshold':   round(r['threshold'], 4),
} for r in all_val_results])

print(summary.to_string(index=False))

best = max(all_val_results, key=lambda r: r['PR-AUC'])
print(f'\nBest model (by PR-AUC): {best["label"]}  PR-AUC={best["PR-AUC"]:.3f}')

              Model  ROC-AUC  PR-AUC  Sensitivity  Specificity  Threshold
           LR (val)    0.839   0.428         0.72        0.850     0.5008
           RF (val)    0.831   0.450         0.64        0.898     0.3652
RF calibrated (val)    0.826   0.432         0.64        0.904     0.0993
          XGB (val)    0.793   0.393         0.56        0.827     0.0961

Best model (by PR-AUC): RF (val)  PR-AUC=0.450


## 7. MIMIC External Test (First and Only Look)

MIMIC is touched here for the first time. No decisions were made using MIMIC data.

In [ ]:
# Use RF calibrated as the best model (consistent with notebook 03)
best_model = rf_calibrated

# Raw eICU threshold on MIMIC
mimic_raw = evaluate_model(best_model, X_mimic, y_mimic,
                           label='RF cal — MIMIC (eICU thr)', threshold=cal_thr)

# Label shift correction
prev_train  = y_train.mean()
prev_target = y_mimic.mean()
log_odds_shift = (np.log(prev_target / (1 - prev_target)) -
                  np.log(prev_train  / (1 - prev_train)))

probs_raw = best_model.predict_proba(X_mimic)[:, 1]
probs_raw = np.clip(probs_raw, 1e-9, 1 - 1e-9)
probs_ls  = 1 / (1 + np.exp(-(np.log(probs_raw / (1 - probs_raw)) + log_odds_shift)))

MIMIC_LS_THR = 0.30
preds_ls = (probs_ls >= MIMIC_LS_THR).astype(int)
tn, fp, fn, tp = confusion_matrix(y_mimic, preds_ls).ravel()

mimic_ls = {
    'label':       'RF cal — MIMIC (label shift)',
    'ROC-AUC':     round(roc_auc_score(y_mimic, probs_ls), 3),
    'PR-AUC':      round(average_precision_score(y_mimic, probs_ls), 3),
    'Sensitivity': round(tp / (tp + fn), 3),
    'Specificity': round(tn / (tn + fp), 3),
    'Threshold':   MIMIC_LS_THR,
    'proba':       probs_ls,
    'y_true':      np.array(y_mimic),
}

print(f'log-odds shift: {log_odds_shift:.4f}  '
      f'(eICU {prev_train:.2%} → MIMIC {prev_target:.2%})')
print()
print(f'MIMIC raw (eICU thr {cal_thr:.4f}):')
print(f'  AUC={mimic_raw["ROC-AUC"]}  Sens={mimic_raw["Sensitivity"]}  Spec={mimic_raw["Specificity"]}')
print(f'\nMIMIC label-shift corrected (thr {MIMIC_LS_THR}):')
print(f'  AUC={mimic_ls["ROC-AUC"]}  Sens={mimic_ls["Sensitivity"]}  Spec={mimic_ls["Specificity"]}')

log-odds shift: 2.2712  (eICU 5.01% → MIMIC 33.82%)

MIMIC raw (eICU thr 0.0993):
  AUC=0.733574879227053  Sens=0.43478260869565216  Spec=0.8666666666666667

MIMIC label-shift corrected (thr 0.3):
  AUC=0.734  Sens=0.674  Spec=0.678


## 8. Save Models

In [ ]:
with open(f'{MODEL_DIR}/rf_calibrated_80.pkl', 'wb') as f:
    pickle.dump(rf_calibrated, f)
with open(f'{MODEL_DIR}/rf_final_80.pkl', 'wb') as f:
    pickle.dump(rf_final, f)

model_params_80 = {
    'rf_best_params':   rf_best_params,
    'rf_threshold':     float(rf_thr),
    'rf_cal_threshold': float(cal_thr),
    'scale_pos':        float(scale_pos),
    'log_odds_shift':   float(log_odds_shift),
    'mimic_threshold':  MIMIC_LS_THR,
}
with open(f'{MODEL_DIR}/model_params_80.json', 'w') as f:
    json.dump(model_params_80, f, indent=2)

threshold_config_80 = {
    'eicu_threshold':  float(cal_thr),
    'mimic_threshold': MIMIC_LS_THR,
    'log_odds_shift':  float(log_odds_shift),
}
with open(f'{MODEL_DIR}/threshold_config_80.pkl', 'wb') as f:
    pickle.dump(threshold_config_80, f)

print('Saved (new files, nothing overwritten):')
for fname in ['rf_calibrated_80.pkl', 'rf_final_80.pkl',
              'model_params_80.json', 'threshold_config_80.pkl']:
    print(f'  {MODEL_DIR}/{fname}')
print()
print(json.dumps(model_params_80, indent=2))

Saved (new files, nothing overwritten):
  /content/drive/MyDrive/AI in Medicine/models/rf_calibrated_80.pkl
  /content/drive/MyDrive/AI in Medicine/models/rf_final_80.pkl
  /content/drive/MyDrive/AI in Medicine/models/model_params_80.json
  /content/drive/MyDrive/AI in Medicine/models/threshold_config_80.pkl

{
  "rf_best_params": {
    "max_depth": 15,
    "min_samples_leaf": 10,
    "n_estimators": 300
  },
  "rf_threshold": 0.3652209333848516,
  "rf_cal_threshold": 0.09927558801545125,
  "scale_pos": 18.96039603960396,
  "log_odds_shift": 2.271184110932317,
  "mimic_threshold": 0.3
}
